<a href="https://colab.research.google.com/github/basit-02/ML-01_Starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/basit-02/ML-01_Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

## 1. Unit of analysis + time window

*Unit of Analysis (Grain): One row represents a single unique content URL (content_id) for a specific client (client_id) aggregated over a one-month calendar observation window (month).   

Time Window: The observation month is 2026-03 (a mid-panel month, deliberately avoiding the final test month of 2026-06).   

Target Table: monthly_content_performance from FlyRank/internship-warehouse.

Output: The analysis hands a human an editorial review queue ranking pages by their observed likelihood to decline, prioritizing which page to fix first.*

In [8]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    hf_token = os.environ.get("HF_TOKEN")

# Connect to DuckDB and configure HF access
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_endpoint='huggingface.co';")
con.execute(f"SET s3_access_key_id='hf';")
con.execute(f"SET s3_secret_access_key='{hf_token}';")

WAREHOUSE_PATH = "hf://datasets/FlyRank/internship-warehouse/monthly_content_performance/*.parquet"

## 2. Fields: feature / label / context / excluded

Context: content_id, client_id, month. These are used for grouping and splitting; never for the model to learn from.   

Label / proxy: is_declining_label. This is the binary target indicating traffic or ranking decay based on observed trends.   

Features (The 5 allowed):

1) impressions: Knowable at the decision moment because Search Console search logs finalize daily prior to queue generation.
2) engagement_rate: Knowable at the decision moment because GA4 sessions are fully computed for the preceding month.
3) scroll_rate: Knowable at the decision moment because tracking beacons record depth interactions during the active month.
4) adjusted_avg_position: Knowable at the decision moment because SERP averages are compiled prior to queue generation (with 0 replaced by 100.0 to reflect unranked status).
5) ai_traffic_pct: Knowable at the decision moment because user-agent access logs identify generative AI bots over the month.

Excluded: health_score and quick_win_flag are hand-written product rules; using them as inputs creates circular logic. trend_pct and trend_direction are excluded because they are the mathematical components used to compute the label.

In [ ]:
# Build the feature frame for the slice
feature_query = f"""
SELECT
    content_id,
    client_id,
    month,
    is_declining_label,
    impressions,
    engagement_rate,
    scroll_rate,
    CASE WHEN avg_position = 0 THEN 100.0 ELSE avg_position END AS adjusted_avg_position,
    ai_traffic_pct,
    -- TRAP FEATURE: Included deliberately to test leakage
    trend_pct AS LEAK_trend_pct
FROM read_parquet('{WAREHOUSE_PATH}')
WHERE month = '2026-03'
  AND ga4_data_available IS TRUE
"""
df_slice = con.execute(feature_query).df()
print(f"Feature frame loaded: {df_slice.shape[0]} rows, {df_slice.shape[1]} columns.")

## 3. Verify it with queries (grain, counts, missing values, windows)

*This section runs the three verification checks for grain, slice boundaries, and data availability, followed by the deliberate leakage trap experiment.*

In [ ]:
# 1. Grain Verification (Should return 0 rows if unit holds)
grain_check = con.execute(f"""
SELECT content_id, client_id, month, COUNT(*) as c
FROM read_parquet('{WAREHOUSE_PATH}')
WHERE month = '2026-03'
GROUP BY content_id, client_id, month
HAVING c > 1
LIMIT 5
""").df()
print(f"Grain Check (duplicates): {len(grain_check)}")

# 2. Slice Bounds Verification (Counts and windows)
span_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(month) AS min_month,
    MAX(month) AS max_month
FROM read_parquet('{WAREHOUSE_PATH}')
WHERE month = '2026-03'
""").df()
print(f"\nSlice Bounds:\n{span_check}")

# 3. Availability Verification (ga4_data_available)
availability_check = con.execute(f"""
SELECT
    COUNT(*) AS total_slice_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS verified_ga4_rows
FROM read_parquet('{WAREHOUSE_PATH}')
WHERE month = '2026-03'
""").df()
print(f"\nAvailability Check:\n{availability_check}")

# --- 4. The Leakage Trap Experiment ---
print("\n--- Leakage Trap Check ---")
df_clean = df_slice.dropna().copy()
features_clean = ['impressions', 'engagement_rate', 'scroll_rate', 'adjusted_avg_position', 'ai_traffic_pct']
features_leaky = features_clean + ['LEAK_trend_pct']
y = df_clean['is_declining_label'].astype(int)

# Leaky Model
clf_leaky = LogisticRegression(max_iter=1000)
clf_leaky.fit(df_clean[features_leaky], y)
preds_leaky = clf_leaky.predict_proba(df_clean[features_leaky])[:, 1]
print(f"Leaky Model ROC-AUC: {roc_auc_score(y, preds_leaky):.4f} (Artificially perfect because of label-derived column)")

# Honest Model
clf_clean = LogisticRegression(max_iter=1000)
clf_clean.fit(df_clean[features_clean], y)
preds_clean = clf_clean.predict_proba(df_clean[features_clean])[:, 1]
print(f"Honest Model ROC-AUC: {roc_auc_score(y, preds_clean):.4f} (Realistic decision-support baseline)")


## 4. Data limits

*Named Limitation: Survivorship bias and attribution lag. Our mid-panel slice (2026-03) paired with the ga4_data_available IS TRUE filter only includes pages that survived long enough to record traffic or sessions. Completely deprecated or 404 pages with zero traffic are excluded from this window, meaning the model is calibrated exclusively on decaying pages that retain residual traffic, rather than catastrophic drop-offs.*

In [ ]:
# Validating missingness patterns to highlight limits
null_check = df_slice[['impressions', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']].isnull().mean() * 100
print("Percentage of missing values per column in our available slice:")
print(null_check.round(2).astype(str) + '%')


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.